# Äquivalenzvergleich: Zeitexpandierter MILP-Solver vs. Dijkstra-Router

Dieses Notebook vergleicht die Ergebnisse des zeitexpandierten MILP-Solvers mit dem implementierten `DijkstraRouter` für verschiedene Gewichtungen und Sendungsszenarien auf `dataset/multimodal_network.json`.

## Hintergrund

**Einzelne Sendung:** Da für eine einzelne Sendung das Problem der optimalen Routenwahl ein klassisches Kürzeste-Weg-Problem auf dem zeitexpandierten Graphen darstellt, müssen sowohl der MILP-Solver als auch der Dijkstra-Router exakt dasselbe Ergebnis liefern.

---

## 1. Setup und Imports

In [2]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights
from freight_routing.model import TimeExpandedFreightRoutingModel
from heuristics.dijkstra_router import DijkstraRouter
from freight_routing.visualization import create_network_map

## 2. Laden des Netzwerks

In [3]:
network_path = PROJECT_ROOT / "dataset/multimodal_network.json"
print(f"Loading network from {network_path}...")
network_data = NetworkDataLoader.from_json(network_path)
network_data.summary()

Loading network from /home/benedikt/Projects/Sustainable_Freight_Mode_Choice/dataset/multimodal_network.json...
Summary NetworkData:
hubs=100
arcs=1664
modes=4


## 3. Definition einer einzelnen Sendung für den ersten Test

In [4]:
shipment_1 = Shipment(
    id="user_shipment_1",
    start_hub="ALG_185",
    end_hub="ORA_186",
    start_time=0,
    deadline=2880,  # 2 Tage Horizont
    max_price=1_000_000.0,
    max_emissions=None,
    weight=2.0,
)

## 4. Basis-Äquivalenztest (Kostenminimierung)

In [5]:
weights_cost = ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)

# MILP
milp_model = TimeExpandedFreightRoutingModel(
    network_data, objective_weights=weights_cost
)
milp_model.build(planning_days=2, shipments=[shipment_1])
milp_res = milp_model.solve([shipment_1])

# Dijkstra
dijkstra_router = DijkstraRouter(network_data, objective_weights=weights_cost)
dijkstra_res = dijkstra_router.solve(shipment_1, planning_days=2)

print("--- Basis-Kostenoptimierung ---")
print(
    f"MILP Cost: {milp_res.total_cost:.2f} EUR | Dijkstra Cost: {dijkstra_res.total_cost:.2f} EUR"
)
milp_route = milp_res.shipment_routes[shipment_1.id]
dijkstra_route = dijkstra_res.shipment_routes[shipment_1.id]
milp_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in milp_route)
dijkstra_path = " -> ".join(
    f"{arc.from_node.hub_id}({arc.mode})" for arc in dijkstra_route
)
print(f"Pfade identisch: {milp_path == dijkstra_path}")

--- Basis-Kostenoptimierung ---
MILP Cost: 1148.64 EUR | Dijkstra Cost: 1148.64 EUR
Pfade identisch: True


## 5. Äquivalenztest mit unterschiedlichen Zielfunktions-Gewichten

1. **Reine Kostenoptimierung**
2. **Reine Emissionsoptimierung**
3. **Reine Zeitoptimierung**
4. **Ausgewogene Optimierung**

**Hinweis zu Einzelszenarien:** Bei reinen Einzelszenarien (z. B. nur Emissions- oder Zeitoptimierung) kann es theoretisch mehrere mathematisch gleichwertige Pfade geben. Da z.B. das Warten an Hubs 0 CO2-Emissionen verursachen kann, ist jeder Abfahrtszeitpunkt für die Emissionen gleichermaßen optimal. Dijkstra wählt den frühesten Start, während der MILP-Solver einen beliebigen gleichwertigen Pfad wählt.

In [6]:
scenarios = [
    ("Reine Kostenoptimierung", ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)),
    ("Reine Emissionsoptimierung", ObjectiveWeights(cost=0.0, emissions=1.0, time=0.0)),
    ("Reine Zeitoptimierung", ObjectiveWeights(cost=0.0, emissions=0.0, time=1.0)),
    ("Ausgewogene Optimierung", ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)),
]

for name, scenario_weights in scenarios:
    print(f"\n=== Szenario: {name} ===")

    # MILP
    m_model = TimeExpandedFreightRoutingModel(
        network_data, objective_weights=scenario_weights
    )
    m_model.build(planning_days=2, shipments=[shipment_1])
    m_res = m_model.solve([shipment_1])

    # Dijkstra
    d_router = DijkstraRouter(network_data, objective_weights=scenario_weights)
    d_res = d_router.solve(shipment_1, planning_days=2)

    if m_res.is_optimal and d_res.is_optimal:
        m_route = m_res.shipment_routes[shipment_1.id]
        d_route = d_res.shipment_routes[shipment_1.id]
        m_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in m_route)
        d_path = " -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in d_route)

        cost_diff = abs(m_res.total_cost - d_res.total_cost)
        time_diff = abs(m_res.total_time - d_res.total_time)
        emissions_diff = abs(m_res.total_emissions - d_res.total_emissions)
        obj_diff = (
            abs(m_res.objective_value - d_res.objective_value)
            if (m_res.objective_value is not None and d_res.objective_value is not None)
            else 0.0
        )

        print(
            f"  Kosten: MILP={m_res.total_cost:.2f} EUR | Dijkstra={d_res.total_cost:.2f} EUR"
        )
        print(
            f"  CO2:    MILP={m_res.total_emissions:.2f} kg  | Dijkstra={d_res.total_emissions:.2f} kg"
        )
        print(
            f"  Dauer:  MILP={m_res.total_time:.2f} min | Dijkstra={d_res.total_time:.2f} min"
        )
        print(f"  Pfade identisch: {m_path == d_path}")

        if "Kosten" in name:
            assert cost_diff < 1e-3, f"Kosten weichen ab: {cost_diff}"
        elif "Emission" in name:
            assert emissions_diff < 1e-3, f"Emissionen weichen ab: {emissions_diff}"
        elif "Zeit" in name:
            assert time_diff < 1e-3, f"Dauer weicht ab: {time_diff}"
        else:
            assert obj_diff < 1e-3, f"Zielfunktionswert weicht ab: {obj_diff}"

        print(f"  [ERFOLG] Äquivalenz verifiziert!")
    else:
        print("  [FEHLER] Einer der Algorithmen fand keine Lösung.")


=== Szenario: Reine Kostenoptimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Reine Emissionsoptimierung ===
  Kosten: MILP=1188.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=521.00 min | Dijkstra=281.00 min
  Pfade identisch: False
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Reine Zeitoptimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!

=== Szenario: Ausgewogene Optimierung ===
  Kosten: MILP=1148.64 EUR | Dijkstra=1148.64 EUR
  CO2:    MILP=104.90 kg  | Dijkstra=104.90 kg
  Dauer:  MILP=281.00 min | Dijkstra=281.00 min
  Pfade identisch: True
  [ERFOLG] Äquivalenz verifiziert!


## 6. Vergleich für mehrere Sendungen mit Konsolidierung

In [7]:
# Definition der Sendungen
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType

shipment_m1 = Shipment(
    id="shipment_m1",
    start_hub="ALG_185",
    end_hub="ANT_1109",
    start_time=0,
    deadline=2880,
    max_price=1000000.0,
    max_emissions=None,
    weight=15.0,
)
shipment_m2 = Shipment(
    id="shipment_m2",
    start_hub="ALG_185",
    end_hub="GEN_1110",
    start_time=0,
    deadline=2880,
    max_price=1000000.0,
    max_emissions=None,
    weight=25.0,
)
multi_shipments = [shipment_m1, shipment_m2]
weights_multi = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)


def format_route(route) -> str:
    segments = []
    for arc in route:
        if arc.arc_type == ArcType.TRANSPORT:
            segments.append(
                f"{arc.from_node.hub_id} -> {arc.to_node.hub_id} ({arc.mode})"
            )
        elif arc.arc_type == ArcType.WAITING:
            segments.append(
                f"Wait@{arc.from_node.hub_id}({arc.mode}, {arc.duration_min}min)"
            )
        elif arc.arc_type == ArcType.TRANSFER:
            segments.append(
                f"Transfer@{arc.from_node.hub_id}({arc.from_node.mode} -> {arc.to_node.mode})"
            )
    return " | ".join(segments)


print("=== 1. MILP-Solver für mehrere Sendungen ===")
milp_multi_model = TimeExpandedFreightRoutingModel(
    network_data, objective_weights=weights_multi
)
milp_multi_model.build(planning_days=2, shipments=multi_shipments)
res_milp_multi = milp_multi_model.solve(multi_shipments)

print(f"Status: {res_milp_multi.status}")
print(f"Kosten: {res_milp_multi.total_cost:.2f} EUR")
print(f"CO2:    {res_milp_multi.total_emissions:.2f} kg")
print(f"Kombinierter Zielfunktionswert: {res_milp_multi.objective_value:.6f}")
print(f"Konsolidiert: {res_milp_multi.is_consolidated}")
print("Routen:")
for s_id, route in res_milp_multi.shipment_routes.items():
    print(f"  {s_id}: {format_route(route)}")

print("\n=== 2. Heuristik-Router (solve_multiple + LNS optimize_multiple) ===")
dijkstra_multi_router = DijkstraRouter(network_data, objective_weights=weights_multi)
res_heur_multi = dijkstra_multi_router.solve_multiple(multi_shipments, planning_days=2)
res_opt_multi = dijkstra_multi_router.optimize_multiple(
    res_heur_multi,
    multi_shipments,
    iterations=20,
    ruin_fraction=0.5,
    planning_days=2,
    seed=42,
)

print(f"Status (Greedy): {res_heur_multi.status}")
print(f"Kosten (Greedy): {res_heur_multi.total_cost:.2f} EUR")
print(f"Status (Optimiert): {res_opt_multi.status}")
print(f"Kosten (Optimiert): {res_opt_multi.total_cost:.2f} EUR")
print(f"CO2 (Optimiert):    {res_opt_multi.total_emissions:.2f} kg")
print(
    f"Kombinierter Zielfunktionswert (Optimiert): {res_opt_multi.objective_value:.6f}"
)
print(f"Konsolidiert: {res_opt_multi.is_consolidated}")
print("Routen (Optimiert):")
for s_id, route in res_opt_multi.shipment_routes.items():
    print(f"  {s_id}: {format_route(route)}")

# Verifikation
assert (
    abs(res_milp_multi.total_cost - res_opt_multi.total_cost) < 1e-3
), "Kosten weichen ab!"
assert (
    abs(res_milp_multi.total_emissions - res_opt_multi.total_emissions) < 1e-3
), "Emissionen weichen ab!"
assert res_opt_multi.is_consolidated is True, "Konsolidierung wurde nicht erkannt!"
print("\n[ERFOLG] Bündelung und mathematische Äquivalenz verifiziert!")

=== 1. MILP-Solver für mehrere Sendungen ===
Status: Optimal
Kosten: 230242.00 EUR
CO2:    38110.32 kg
Kombinierter Zielfunktionswert: 0.235618
Konsolidiert: True
Routen:
  shipment_m1: Wait@ALG_185(air, 300min) | Wait@ALG_185(air, 180min) | Wait@ALG_185(air, 60min) | ALG_185 -> PAR_3672 (air) | Wait@PAR_3672(air, 7min) | Wait@PAR_3672(air, 4min) | Wait@PAR_3672(air, 124min) | Wait@PAR_3672(air, 60min) | Wait@PAR_3672(air, 45min) | Wait@PAR_3672(air, 15min) | PAR_3672 -> ANT_1109 (air)
  shipment_m2: Wait@ALG_185(air, 300min) | Wait@ALG_185(air, 180min) | Wait@ALG_185(air, 60min) | ALG_185 -> PAR_3672 (air) | Wait@PAR_3672(air, 7min) | Wait@PAR_3672(air, 4min) | Wait@PAR_3672(air, 124min) | Wait@PAR_3672(air, 60min) | Wait@PAR_3672(air, 45min) | Wait@PAR_3672(air, 15min) | Wait@PAR_3672(air, 9min) | Wait@PAR_3672(air, 9min) | Wait@PAR_3672(air, 57min) | Wait@PAR_3672(air, 38min) | Wait@PAR_3672(air, 7min) | Wait@PAR_3672(air, 60min) | Transfer@PAR_3672(air -> road) | PAR_3672 -> GEN_11

## 7. Vergleich auf hoher Konsolidierungsdichte

In [8]:
# Laden des mittleren Netzwerks
medium_network_data = NetworkDataLoader.from_json("../dataset/medium_network.json")

import random
import time

# Generierung von 50 zufälligen Sendungen
random.seed(42)
N_SHIPMENTS = 50

hubs_list = list(medium_network_data.hubs.keys())
test_shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(hubs_list)
    dest = random.choice(hubs_list)
    while dest == start:
        dest = random.choice(hubs_list)

    test_shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=0,
            deadline=2880,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(1, 10)),
        )
    )

weights_medium = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)

print(f"{N_SHIPMENTS} Sendungen generiert.")

print("=== 1. MILP-Solver ===")
t0 = time.time()
milp_medium = TimeExpandedFreightRoutingModel(
    medium_network_data, objective_weights=weights_medium
)
milp_medium.build(planning_days=2, shipments=test_shipments)
milp_medium.summary()
res_milp_medium = milp_medium.solve(test_shipments)
t_milp = time.time() - t0

print(f"Status: {res_milp_medium.status}")
print(f"Kosten: {res_milp_medium.total_cost:.2f} EUR")
print(f"CO2:    {res_milp_medium.total_emissions:.2f} kg")
print(f"Kombinierter Zielfunktionswert: {res_milp_medium.objective_value:.6f}")
print(f"Konsolidiert: {res_milp_medium.is_consolidated}")
print(f"Rechenzeit MILP: {t_milp:.2f} s")

print("\n=== 2. Heuristik-Router (solve_multiple + LNS optimize_multiple) ===")
t0 = time.time()
dijkstra_medium_router = DijkstraRouter(
    medium_network_data, objective_weights=weights_medium
)
res_heur_medium = dijkstra_medium_router.solve_multiple(test_shipments, planning_days=2)
res_opt_medium = dijkstra_medium_router.optimize_multiple(
    res_heur_medium,
    test_shipments,
    iterations=50,
    ruin_fraction=0.2,
    planning_days=2,
    seed=42,
)
t_heur = time.time() - t0

print(f"Status (Greedy): {res_heur_medium.status}")
print(f"Kosten (Greedy): {res_heur_medium.total_cost:.2f} EUR")
print(f"Status (Optimiert): {res_opt_medium.status}")
print(f"Kosten (Optimiert): {res_opt_medium.total_cost:.2f} EUR")
print(f"CO2 (Optimiert):    {res_opt_medium.total_emissions:.2f} kg")
print(
    f"Kombinierter Zielfunktionswert (Optimiert): {res_opt_medium.objective_value:.6f}"
)
print(f"Konsolidiert: {res_opt_medium.is_consolidated}")
print(f"Rechenzeit Heuristik (gesamt): {t_heur:.2f} s")
if not res_opt_medium.is_optimal:
    print("Diagnostics:")
    for d in res_opt_medium.diagnostics:
        print(d)

# Verifikation
assert res_opt_medium.is_consolidated is True, "Konsolidierung wurde nicht erkannt!"

50 Sendungen generiert.
=== 1. MILP-Solver ===
Summary TimeExpandedFreightRoutingModel:
planning_days=2
planning_horizon_min=2880
nodes=2128
arcs=3576
  - transport_arcs=1004
  - transfer_arcs=484
  - waiting_arcs=2088
Status: Optimal
Kosten: 1256835.69 EUR
CO2:    207672.11 kg
Kombinierter Zielfunktionswert: 5.847327
Konsolidiert: True
Rechenzeit MILP: 44.43 s

=== 2. Heuristik-Router (solve_multiple + LNS optimize_multiple) ===
Status (Greedy): Optimal
Kosten (Greedy): 1257983.23 EUR
Status (Optimiert): Optimal
Kosten (Optimiert): 1257833.23 EUR
CO2 (Optimiert):    207235.38 kg
Kombinierter Zielfunktionswert (Optimiert): 6.508212
Konsolidiert: True
Rechenzeit Heuristik (gesamt): 1.19 s
